## An analysis of NYC Block Party data

In [229]:
import pandas as pd
from plotnine import *
import matplotlib.pyplot as plt

plt.rcParams['svg.fonttype'] = 'none'

#### Loading in the data I downloaded from [here](https://data.cityofnewyork.us/City-Government/NYC-Permitted-Event-Information-Historical/bkfu-528j/explore/query/SELECT%0A%20%20%60event_id%60%2C%0A%20%20%60event_name%60%2C%0A%20%20%60start_date_time%60%2C%0A%20%20%60end_date_time%60%2C%0A%20%20%60event_agency%60%2C%0A%20%20%60event_type%60%2C%0A%20%20%60event_borough%60%2C%0A%20%20%60event_location%60%2C%0A%20%20%60event_street_side%60%2C%0A%20%20%60street_closure_type%60%2C%0A%20%20%60community_board%60%2C%0A%20%20%60police_precinct%60%0AWHERE%20caseless_eq%28%60event_type%60%2C%20%22Block%20Party%22%29/page/filter) (the NYC Open Data Portal)

In [230]:
df_old = pd.read_csv('data/block_party_permits.csv')

There are about 2700 rows that have NA values for the community board (about 2.3% of the total dataset). Rather than trying some sort of complicated analysis of the addresses to map onto another library of community boards, I'm deciding to just drop these rows.

In [231]:
df_old[df_old['Community Board'].isna()].shape

(2753, 12)

#### Making the column names cleaner to reference

In [232]:
df = df_old.rename(columns={'Event ID' : "event_id",
                   "Event Name": 'event_name',
                   "Start Date/Time": 'start_date_time',
                   "End Date/Time": 'end_date_time',
                   'Event Agency': 'event_agency',
                   'Event Type': 'type',
                   'Event Borough': 'borough',
                   'Event Location': 'location',
                   'Event Street Side': 'street_side',
                   'Street Closure Type': 'street_closure_type',
                   'Community Board': 'community_board',
                   'Police Precinct': 'police_precinct'})

##### Now I need to make these column types match what they *should* be (not all objects)

In [233]:
df.dtypes

event_id                int64
event_name             object
start_date_time        object
end_date_time          object
event_agency           object
type                   object
borough                object
location               object
street_side            object
street_closure_type    object
community_board        object
police_precinct        object
dtype: object

##### To make these columns cleaned for my analysis, I had to remove the commas that the cells had and remove any insances of spaces following data. I also came across several rows that had multiple community boards in the cell (listed like "4 18 26 ", for example). In this case, I chose to just always select the first element when splitting by spaces.

In [234]:
df['community_board'] = df['community_board'].str.replace(',','')
df['community_board'] = df['community_board'].str.split(" ").str[0]
df = df.dropna(subset=['community_board'])
df['community_board'] = df['community_board'].astype('int64')

df['police_precinct'] = df['police_precinct'].str.replace(',','')
df['police_precinct'] = df['police_precinct'].str.split(" ").str[0]
df = df.dropna(subset=['police_precinct'])
df['police_precinct'] = df['police_precinct'].astype('int64')

##### I also made the date_time columns actually datetime columns

In [235]:
df['start_date_time'] = df['start_date_time'].astype('datetime64[ns]')
df['end_date_time'] = df['end_date_time'].astype('datetime64[ns]')

In [236]:
df['start_year'] = df['start_date_time'].dt.year
df['start_month'] = df['start_date_time'].dt.month
df['start_day'] = df['start_date_time'].dt.day

df['end_year'] = df['end_date_time'].dt.year
df['end_month'] = df['end_date_time'].dt.month
df['end_day'] = df['end_date_time'].dt.day

*this is not AI even though I'm using print()... I just like having things nicely formatted and easily readable*

In [237]:
print("dtypes:")
print(df.dtypes)
print("=== === === === === ===")
print("Shape: ", df.shape)
print("=== === === === === ===")
df.head()

dtypes:
event_id                        int64
event_name                     object
start_date_time        datetime64[ns]
end_date_time          datetime64[ns]
event_agency                   object
type                           object
borough                        object
location                       object
street_side                    object
street_closure_type            object
community_board                 int64
police_precinct                 int64
start_year                      int32
start_month                     int32
start_day                       int32
end_year                        int32
end_month                       int32
end_day                         int32
dtype: object
=== === === === === ===
Shape:  (114310, 18)
=== === === === === ===


,event_id,event_name,start_date_time,end_date_time,event_agency,type,borough,location,street_side,street_closure_type,community_board,police_precinct,start_year,start_month,start_day,end_year,end_month,end_day
0,378815,SIS Admissions Fair,2017-11-18 12:00:00,2017-11-18 16:00:00,Street Activity Permit Office,Block Party,Bronx,MANIDA STREET between LAFAYETTE AVENUE and SP...,South,Full Street Closure,2,41,2017,11,18,2017,11,18
1,372825,Little North Pole,2017-11-12 14:00:00,2017-11-12 20:00:00,Street Activity Permit Office,Block Party,Queens,NEPONSIT AVENUE between BEACH 144 STREET and...,Full,Full Street Closure,14,100,2017,11,12,2017,11,12
2,376296,3rd Annual Community Thanksgiving,2017-11-24 13:00:00,2017-11-24 18:00:00,Street Activity Permit Office,Block Party,Bronx,GOBLE PLACE between INWOOD AVENUE and MACOMBS...,Both,Sidewalk and Street Closure,4,44,2017,11,24,2017,11,24
3,364624,Fall Fair Festival,2017-11-12 11:00:00,2017-11-12 16:00:00,Street Activity Permit Office,Block Party,Manhattan,WEST 100 STREET between BROADWAY and WEST EN...,Full,Full Street Closure,7,24,2017,11,12,2017,11,12
4,378815,SIS Admissions Fair,2017-11-18 12:00:00,2017-11-18 16:00:00,Street Activity Permit Office,Block Party,Bronx,MANIDA STREET between LAFAYETTE AVENUE and SP...,South,Full Street Closure,2,41,2017,11,18,2017,11,18


### Making `location` column workable

##### The current format for the `location` variable is, for example, " MANIDA STREET between LAFAYETTE AVENUE and SPOFFORD AVENUE". This is not super helpful for, hopefully, eventually mapping. I need to first isola

In [238]:
df_test = df

In [239]:
df_test['loc_test'] = df_test['location'].str.lower()

In [240]:
df_test.head()

,event_id,event_name,start_date_time,end_date_time,event_agency,type,borough,location,street_side,street_closure_type,community_board,police_precinct,start_year,start_month,start_day,end_year,end_month,end_day,loc_test
0,378815,SIS Admissions Fair,2017-11-18 12:00:00,2017-11-18 16:00:00,Street Activity Permit Office,Block Party,Bronx,MANIDA STREET between LAFAYETTE AVENUE and SP...,South,Full Street Closure,2,41,2017,11,18,2017,11,18,manida street between lafayette avenue and sp...
1,372825,Little North Pole,2017-11-12 14:00:00,2017-11-12 20:00:00,Street Activity Permit Office,Block Party,Queens,NEPONSIT AVENUE between BEACH 144 STREET and...,Full,Full Street Closure,14,100,2017,11,12,2017,11,12,neponsit avenue between beach 144 street and...
2,376296,3rd Annual Community Thanksgiving,2017-11-24 13:00:00,2017-11-24 18:00:00,Street Activity Permit Office,Block Party,Bronx,GOBLE PLACE between INWOOD AVENUE and MACOMBS...,Both,Sidewalk and Street Closure,4,44,2017,11,24,2017,11,24,goble place between inwood avenue and macombs...
3,364624,Fall Fair Festival,2017-11-12 11:00:00,2017-11-12 16:00:00,Street Activity Permit Office,Block Party,Manhattan,WEST 100 STREET between BROADWAY and WEST EN...,Full,Full Street Closure,7,24,2017,11,12,2017,11,12,west 100 street between broadway and west en...
4,378815,SIS Admissions Fair,2017-11-18 12:00:00,2017-11-18 16:00:00,Street Activity Permit Office,Block Party,Bronx,MANIDA STREET between LAFAYETTE AVENUE and SP...,South,Full Street Closure,2,41,2017,11,18,2017,11,18,manida street between lafayette avenue and sp...
